# 10. 用 π0.5 微调 OMY 机械臂并在 MuJoCo 中评测

这份 Notebook［交互式笔记本］接续 [1.collect_data.ipynb](1.collect_data.ipynb) 和 [2.visualize_data-omy.ipynb](2.visualize_data-omy.ipynb)，完成：检查示教 → 迁移数据 → 补充分位数 → 五步训练检查 → 正式微调 → 离线动作对比 → 仿真抓放。

对象是 **ROBOTIS OMY 六轴机械臂与夹爪的单杯抓放场景** `asset/example_scene_y.xml`。使用独立的 **Python 3.12 / LeRobot 0.6.1 / PyTorch 2.7.1 CUDA 12.8** 环境，本章第 1～9 节继续使用各自的旧环境。不要在旧环境内升级依赖，也不要用旧 `train_model.py` 启动本流程。

| 输入/输出 | 本流程定义 |
| --- | --- |
| 两路图像 | 外部相机、腕部相机，采集为 256×256 RGB［红绿蓝彩色图］ |
| `observation.state` | 6 维末端位姿 `[x,y,z,roll,pitch,yaw]`，位置为米，旋转为弧度 |
| `action` | 7 维绝对目标关节角 `[joint1,…,joint6,gripper]`，角度为弧度，夹爪 0 开、1 闭 |
| 任务 | 每帧的 `task` 语言指令，来自原数据任务索引 |
| `obj_init` | 杯子与盘子的两个初始位置，仅用于回放和仿真重置 |
| 频率 | 20 Hz［每秒 20 次控制］，保留采集频率 |

**适用范围：** 第 5 节语言场景的六维状态是关节角，物体和重置接口也不同，不能直接套用这里的转换和评测。这里使用第 1 节末端位姿数据。模型内部处理动作填充（默认 32 维）；无需手工把七维动作改为 32 维。末端位姿与关节目标不是同一种量，因此明确设置 `use_relative_actions=False`。

π0.5 的通用预训练权重是微调起点，还不是已学会 OMY 抓放的策略。默认只训练动作专家并冻结视觉编码器，批量大小为 1，启用梯度检查点。官方完整训练示例使用大显存硬件；这些设置也不保证能在 6 GB 显存上训练。建议在实验室的 Linux［操作系统］工作站上执行，以五步检查时的实际峰值显存为准。数据、模型缓存和输出放到大容量磁盘。


## 1. 创建独立环境

以下命令在 **本章目录的终端**执行一次，然后在笔记本菜单选择 `OMY pi05 (Python 3.12)` 内核。先安装与显卡驱动匹配的 PyTorch［张量计算框架］预编译包，再安装本流程依赖。需要系统已有 `ffmpeg`［视频编码工具］和图形桌面；本章 MuJoCo［物理仿真器］窗口依赖显示服务。无桌面服务器可用 `xvfb-run`［虚拟显示工具］启动笔记本服务。

```bash
conda create -n omy-pi05 python=3.12 -y
conda activate omy-pi05
python -m pip install torch==2.7.1 torchvision==0.22.1 --index-url https://download.pytorch.org/whl/cu128
python -m pip install -r requirements_pi05.txt
python -m ipykernel install --user --name omy-pi05 --display-name 'OMY pi05 (Python 3.12)'
jupyter lab .
```

工作站如使用其他环境管理工具，可将环境放在 `/data/Data14TB/envs/`。开始前在终端设置工作目录，例如：

```bash
export OMY_PI05_ROOT=/data/Data14TB/robotics_shared/experiments/omy-pi05
export HF_HOME=/data/Data14TB/robotics_shared/cache/huggingface
```

场景资源沿用本章安装步骤：将 `asset/objaverse/plate_11.zip` 解压到同目录。不要修改原采集数据目录。首次下载预训练权重需要网络和足够磁盘；短步检查、正式训练各自保存完整模型与训练状态，合计空间可能达到数十 GB。


In [ ]:
import os
import sys
import json
import shlex
import subprocess
from pathlib import Path
from datetime import datetime

CHAPTER = Path.cwd().resolve()
assert (CHAPTER / "pi05_omy_utils.py").is_file(), "Start Jupyter in this chapter directory"
SOURCE = CHAPTER / "demo_data"  # Newly collected chapter-1 data; not demo_data_language.
RUN_ROOT = Path(os.environ.get("OMY_PI05_ROOT", CHAPTER / "pi05_outputs")).resolve()
V3_ROOT = RUN_ROOT / "omy_v3"
REPO_ID = "local/omy-pi05-20hz"
RUN_ID = datetime.now().strftime("%Y%m%d-%H%M%S")
SMOKE_OUTPUT = RUN_ROOT / f"smoke-{RUN_ID}"
TRAIN_OUTPUT = RUN_ROOT / f"train-{RUN_ID}"
BASE_MODEL = os.environ.get("OMY_PI05_PRETRAINED", "lerobot/pi05_base")
SMOKE_STEPS = 5
CHUNK_SIZE = 50
N_ACTION_STEPS = 5
TRAIN_STEPS = 10000
BATCH_SIZE = 1

# Set True only after checking the current collector and replaying your demonstrations.
ACTION_LABELS_VERIFIED = False
RUN_SMOKE = False
RUN_TRAIN = False

RUN_ROOT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(RUN_ROOT / "hf_cache"))
from pi05_omy_utils import (check_version, read_v21, prepare_dataset,
                            verify_alignment, training_command, load_policy, rollout)
print("source:", SOURCE)
print("outputs:", RUN_ROOT)


In [ ]:
import importlib.metadata as metadata
import torch

check_version()
assert sys.version_info[:2] == (3, 12), sys.version
assert torch.__version__.split("+")[0] == "2.7.1", torch.__version__
assert torch.cuda.is_available(), "Select the CUDA workstation kernel"
assert torch.cuda.is_bf16_supported(), "This configuration requires BF16 support"
assert metadata.version("mujoco") == "3.1.6"
free, total = torch.cuda.mem_get_info()
print("Python:", sys.version.split()[0])
print("LeRobot:", metadata.version("lerobot"))
print("PyTorch / CUDA:", torch.__version__, torch.version.cuda)
print("device:", torch.cuda.get_device_name(0))
print(f"free / total VRAM: {free/2**30:.1f} / {total/2**30:.1f} GiB")
if free < 20 * 2**30:
    print("Low free VRAM: reduce workload or use the larger workstation before training")


## 2. 采集、回放并确认动作标签

先在旧版采集环境运行第 1 节，增加 `NUM_DEMO`，采集不同杯子位置的完整成功示教，再用第 2 节回放。建议先积累 30～50 个回合检查流程，所需数量最终取决于任务覆盖和评测结果。

**2026-10-02 前的旧采集入口曾把测量关节状态存成动作标签。格式转换不能恢复丢失的目标命令。** 请使用修复后的 `get_commanded_joint_action()` 接口重新采集，或确认数据来源已经采用该接口。仓库 `demo_data_example` 是历史结构示例，不作为这里的训练默认值。

下面的检查可以发现缺失文件、错误维度、时间戳、夹爪范围、图像和回合索引问题，**不能仅凭数组判断动作标签来源或状态含义**。确认第 1 节末端位姿、实际下发的关节目标与回放均正确后，把配置单元的 `ACTION_LABELS_VERIFIED` 设为 `True`。


In [ ]:
info, episodes = read_v21(SOURCE)
print({key: info[key] for key in ["codebase_version", "fps", "total_episodes", "total_frames"]})
print("tasks:", sorted({row["task"] for rows in episodes for row in rows}))
print("first state:", episodes[0][0]["observation.state"])
print("first action:", episodes[0][0]["action"])
print("action semantics must be verified from the collector and replay")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pi05_omy_utils import _image

first = episodes[0][0]
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, key in zip(axes, ["observation.image", "observation.wrist_image"]):
    ax.imshow(_image(first[key], SOURCE.resolve(), (256, 256, 3)))
    ax.set_title(key)
    ax.axis("off")
plt.show()
actions = np.stack([row["action"] for row in episodes[0]])
plt.figure(figsize=(10, 3))
plt.plot(np.arange(len(actions))/20, actions)
plt.xlabel("simulation time (s)")
plt.ylabel("joint target (rad) / normalized gripper")
plt.legend([f"joint{i}" for i in range(1, 7)] + ["gripper"], ncol=4)
plt.show()


## 3. 转换为独立的 LeRobot v3 数据

本章 v2.1 图像存储在 Parquet［列式数据文件］中。辅助程序用新版官方数据写入接口逐回合重建 v3 数据，保留图像、六维状态、七维动作、任务、初始位置和 20 Hz［每秒 20 次控制］频率，仅规范相机键名与索引。

目标目录必须不存在；重复执行时选择新目录，不会自动删除原数据或已有转换结果。已有本流程的转换结果可跳过转换单元，直接读取 `omy_manifest.json` 后继续校验。

三个以上回合时，按完整回合随机留出约 20% 作评测，固定种子为 42。仅用训练回合计算状态和动作的精确 `q01/q99`［1%/99% 分位数］及均值、标准差等统计量，不把留出回合用于归一化。一两条示教只适合接口检查或过拟合调试，不报告泛化成功率。


In [ ]:
manifest = prepare_dataset(SOURCE, V3_ROOT, REPO_ID,
                           labels_verified=ACTION_LABELS_VERIFIED, split_seed=42)
print(json.dumps(manifest, indent=2, ensure_ascii=False))


In [ ]:
# This cell can also reopen an existing conversion produced by this notebook.
manifest = json.loads((V3_ROOT / "omy_manifest.json").read_text(encoding="utf-8"))
print(verify_alignment(V3_ROOT, chunk_size=CHUNK_SIZE))
stats = json.loads((V3_ROOT / "meta/stats.json").read_text(encoding="utf-8"))
for key in ["observation.state", "action"]:
    print(key, "q01:", stats[key]["q01"], "q99:", stats[key]["q99"])
print("train episodes:", manifest["train_episodes"])
print("held-out episodes:", manifest["heldout_episodes"])


## 4. 检查模型访问并准备训练命令

微调默认使用 [lerobot/pi05_base](https://huggingface.co/lerobot/pi05_base)。已有兼容的本地预训练目录时，在启动前设置 `OMY_PI05_PRETRAINED`，或直接修改配置单元的 `BASE_MODEL`，可复用磁盘上的权重。该目录需要模型、LeRobot 配置以及输入、输出处理器配置；原始 OpenPi 目录需要先准备这些适配文件。输入处理还需要 `google/paligemma-3b-pt-224` 的分词器。若提示访问受限，先在其模型页面接受许可，再在终端执行 `hf auth login`［模型平台账号登录］；不要把访问令牌写进笔记本。

训练通过本章 `pi05_omy_train.py` 调用官方训练入口，并检查全部预训练参数加载成功。固定版本的官方加载器可能在读取权重失败后返回随机初始化模型，本章会把这种情况作为错误停止；推理加载也使用同一检查。

相机键映射为 `observation.images.image` 和 `observation.images.image2`，模型特征从转换后的元数据取得。语言与量化后的末端状态一起送入分词器。训练和推理都使用官方输入、输出处理器，不能只读取权重后直接把原始数组送进模型。

`chunk_size=50`［预测 50 步动作块］保持预训练默认值；`n_action_steps=5`［连续执行其中 5 步］使策略约每 0.25 秒仿真时间重新观察。短步检查只验证数据、前向、反向、保存与加载链路，不代表学会抓放。


In [ ]:
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer

# Only config/tokenizer files are requested here; weights load when training starts.
if Path(BASE_MODEL).is_dir():
    required = ["config.json", "model.safetensors", "policy_preprocessor.json", "policy_postprocessor.json"]
    assert all((Path(BASE_MODEL) / name).is_file() for name in required), "Incomplete local pretrained directory"
    model_config = json.loads((Path(BASE_MODEL) / "config.json").read_text(encoding="utf-8"))
else:
    model_config = json.loads(Path(hf_hub_download(BASE_MODEL, "config.json")).read_text(encoding="utf-8"))
assert model_config["type"] == "pi05", model_config.get("type")
tokenizer = AutoTokenizer.from_pretrained("google/paligemma-3b-pt-224")
print("model configuration and tokenizer access: OK")
smoke_cmd = training_command(V3_ROOT, SMOKE_OUTPUT, steps=SMOKE_STEPS,
    batch_size=BATCH_SIZE, chunk_size=CHUNK_SIZE, n_action_steps=N_ACTION_STEPS,
    pretrained=BASE_MODEL)
train_cmd = training_command(V3_ROOT, TRAIN_OUTPUT, steps=TRAIN_STEPS,
    batch_size=BATCH_SIZE, chunk_size=CHUNK_SIZE, n_action_steps=N_ACTION_STEPS,
    pretrained=BASE_MODEL)
print(f"{SMOKE_STEPS}-step check:\n", shlex.join(smoke_cmd))
print("full training:\n", shlex.join(train_cmd))


In [ ]:
def run_training(command, log_path):
    # Stream logs; interrupting the cell stops only the process started by this cell.
    with Path(log_path).open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, cwd=CHAPTER, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                log.write(line)
                log.flush()
                print(line, end="")
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
    if returncode:
        raise RuntimeError(f"Training failed ({returncode}); inspect {log_path}")

if RUN_SMOKE:
    run_training(smoke_cmd, RUN_ROOT / f"smoke-{RUN_ID}.log")
    smoke_checkpoint = SMOKE_OUTPUT / "checkpoints/last/pretrained_model"
    assert (smoke_checkpoint / "config.json").is_file(), smoke_checkpoint
else:
    print("Set RUN_SMOKE=True in the configuration cell when ready")


## 5. 正式微调与断点恢复

五步检查正常结束后，检查日志中的损失、梯度范数、峰值显存，以及最后保存的完整检查点。将 `RUN_TRAIN` 设为 `True`，重新生成正式训练命令并执行下面单元。

初始参数为 10000 步、批量大小 1，每 2000 步保存。数据量小时先观察是否过拟合，最终以留出场景抓放成功率决定模型是否可用。训练步数和批量大小不是已经验证的最优值。

中断后可以用保存的完整训练配置恢复模型、优化器和调度器。恢复命令形如：

```bash
python -m pi05_omy_train --config_path=/absolute/path/train-*/checkpoints/last/pretrained_model/train_config.json --resume=true
```

请把占位路径替换成实际文件，不要把新训练输出目录指向已有结果。短步检查保存的模型只能用于链路检查；下面评测默认读取正式训练目录。


In [ ]:
if RUN_TRAIN:
    # Rebuild to ensure the output directory is still new before launching.
    train_cmd = training_command(V3_ROOT, TRAIN_OUTPUT, steps=TRAIN_STEPS,
        batch_size=BATCH_SIZE, chunk_size=CHUNK_SIZE, n_action_steps=N_ACTION_STEPS,
        pretrained=BASE_MODEL)
    run_training(train_cmd, RUN_ROOT / f"train-{RUN_ID}.log")
else:
    print("Set RUN_TRAIN=True after the five-step check passes")


## 6. 加载正式检查点并对比留出动作

如在终端训练或重新打开笔记本，先把 `CHECKPOINT` 改成实际的 `pretrained_model` 目录。加载时同时读取保存的归一化、分词与设备处理配置，不重新估计统计量。

离线比较的是首步的六个关节目标及夹爪命令。两者单位不同，分别报告误差。离线误差不能代替仿真成功率；这里最多检查八帧，便于先检查输出形状和数值。


In [ ]:
CHECKPOINT = TRAIN_OUTPUT / "checkpoints/last/pretrained_model"
policy, preprocessor, postprocessor = load_policy(CHECKPOINT)
print("state:", policy.config.input_features["observation.state"])
print("action:", policy.config.output_features["action"])


In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset
import pyarrow.parquet as pq

assert manifest["heldout_episodes"], "Collect at least three demos for a held-out evaluation"
dataset = LeRobotDataset(REPO_ID, root=V3_ROOT,
    delta_timestamps={"action": [i / 20 for i in range(policy.config.chunk_size)]})
index_rows = []
for path in sorted((V3_ROOT / "data").rglob("*.parquet")):
    index_rows.extend(pq.read_table(path).select(["index", "episode_index", "frame_index", "obj_init"]).to_pylist())
eval_indices = [r["index"] for r in index_rows if r["episode_index"] in manifest["heldout_episodes"]]
chosen = np.linspace(0, len(eval_indices) - 1, min(8, len(eval_indices)), dtype=int)
errors = []
for offset in chosen:
    sample = dataset[eval_indices[offset]]
    observation = {key: sample[key] for key in ["observation.images.image", "observation.images.image2", "observation.state", "task"]}
    policy.reset()
    preprocessor.reset()
    postprocessor.reset()
    with torch.inference_mode():
        predicted = postprocessor(policy.predict_action_chunk(preprocessor(observation)))[0, 0].cpu().numpy()
    target = sample["action"][0].numpy()
    assert predicted.shape == (7,) and np.isfinite(predicted).all()
    errors.append(np.abs(predicted - target))
errors = np.stack(errors)
print("joint target MAE (rad):", errors[:, :6].mean(axis=0))
print("gripper MAE:", errors[:, 6].mean())


## 7. 在 OMY 单杯场景闭环评测

训练环境没有登记为官方评测环境，因此训练时关闭官方环境评测，使用本节的本章场景。先按留出回合 `obj_init` 重置杯子、盘子位置，再让策略独立执行抓放，记录真实成功判定。

每次采集当前两路图像与末端位姿，预测绝对关节目标，用 `action_type='joint_angle'` 下发。每个控制周期推进 0.05 秒物理仿真，频率与 20 Hz［每秒 20 次控制］示教一致。推理耗时可能使墙上时钟下的播放更慢，但不会通过改数据频率来加速。此处同步控制不等于满足真机实时控制要求。

环境会裁剪夹爪命令并限制开合速率，所以同时保存模型原始预测和实际下发动作。机械臂控制沿用本章简化仿真接口；结果用于该场景的策略检查。每回合最多 600 步，即 30 秒仿真时间。视频、动作与成功率写到新的评测目录，不覆盖旧结果。


In [ ]:
import cv2
assert os.environ.get("DISPLAY"), "Run on the Ubuntu desktop or launch Jupyter with xvfb-run"
from mujoco_env.y_env import SimpleEnv
EVAL_OUTPUT = RUN_ROOT / f"eval-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
EVAL_OUTPUT.mkdir(exist_ok=False)
EVAL_EPISODES = manifest["heldout_episodes"][:5]
assert EVAL_EPISODES, "No held-out episodes; collect more demonstrations"
records = []
env = SimpleEnv(xml_path=str(CHAPTER / "asset/example_scene_y.xml"),
                action_type="joint_angle", state_type="ee_pose", seed=42)
try:
    for episode in EVAL_EPISODES:
        initial_row = next(r for r in index_rows if r["episode_index"] == episode and r["frame_index"] == 0)
        initial = np.asarray(initial_row["obj_init"], dtype=np.float32)
        env.reset(seed=42 + episode)
        env.set_obj_pose(initial[:3], initial[3:])
        for _ in range(100):
            env.step_env()
        task = dataset[initial_row["index"]]["task"]
        torch.manual_seed(42 + episode)
        result = rollout(policy, preprocessor, postprocessor, env, task=task, fps=20, max_steps=600)
        frames = result.pop("frames")
        video_path = EVAL_OUTPUT / f"episode-{episode:03d}.mp4"
        height, width = frames[0].shape[:2]
        writer = cv2.VideoWriter(str(video_path), cv2.VideoWriter_fourcc(*"mp4v"), 20, (width, height))
        assert writer.isOpened(), "Video encoder could not open the output"
        try:
            for frame in frames:
                writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
        finally:
            writer.release()
        result["predicted_actions"] = [a.tolist() for a in result["predicted_actions"]]
        result["applied_actions"] = [a.tolist() for a in result["applied_actions"]]
        result.update({"episode": episode, "seed": 42 + episode, "task": task, "video": str(video_path), "checkpoint": str(CHECKPOINT)})
        (EVAL_OUTPUT / f"episode-{episode:03d}.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
        records.append(result)
        print("episode:", episode, "success:", result["success"], "steps:", result["steps"])
finally:
    env.env.close_viewer()

summary = {"evaluated_episodes": EVAL_EPISODES,
           "successes": sum(r["success"] for r in records), "episodes": len(records),
           "success_rate": sum(r["success"] for r in records) / len(records),
           "checkpoint": str(CHECKPOINT)}
(EVAL_OUTPUT / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(summary)


## 检查与常见问题

- 在独立环境中运行 `python -m unittest -v test_pi05_omy`。检查使用真实新版数据写入和读取接口，覆盖图像/动作保真、打乱行顺序、回合末尾动作填充、训练回合分位数、原数据保留和 20 Hz［每秒 20 次控制］下发时序。
- 目标目录已经存在：改用新的 `V3_ROOT`，或跳过转换单元并读取已有清单；不要删除原示教。
- 没有 `q01/q99`［1%/99% 分位数］：确认使用本流程转换后的数据，检查 `meta/stats.json`；不要把均值和标准差改名当作分位数。
- 图像或模型无法读取：检查数据是否完整下载，访问许可是否已接受，以及独立内核是否已选中。
- 训练显存不足：先保持批量大小 1 与冻结设置，在大显存工作站运行五步检查。缩短动作块会改变时域覆盖，应同步检查动作块配置和评测执行步数。
- 低离线误差但仿真失败：先回放留出示教，检查状态定义、相机顺序、绝对关节动作、初始位置和夹爪方向；再增加示教覆盖与训练。不要根据损失下降宣称抓放复现成功。

本流程的支持依据：官方 [π0.5 使用说明](https://huggingface.co/docs/lerobot/main/en/pi05)、[0.6.1 固定版依赖](https://github.com/huggingface/lerobot/blob/v0.6.1/pyproject.toml)、[π0.5 配置](https://github.com/huggingface/lerobot/blob/v0.6.1/src/lerobot/policies/pi05/configuration_pi05.py) 和 [官方训练入口](https://github.com/huggingface/lerobot/blob/v0.6.1/src/lerobot/scripts/lerobot_train.py)。流程固定使用 0.6.1，最新文档的其他版本命令不一定兼容。

运行本笔记本不会自动上传数据、模型或实验结果。正式复现记录应注明示教回合数、训练/留出划分、训练设置、检查点和实际成功率。
